# ML-08 ? Capstone Modeling Lane

This notebook builds a first learned model for the starter content-refresh lane. The task is **ranking pages for review when the observed 90-day trend is declining**. The label is an observational proxy, not proof that a refresh will cause recovery.

The notebook uses the bundled anonymized starter CSV and the Week 4 rule baseline. It deliberately excludes `trend_direction`, `trend_pct`, IDs, and other label-derived fields from the feature matrix.

## 1. Method choice and why

I will compare a readable **Logistic Regression** model with a shallow **Decision Tree** and a constrained **Random Forest**. This is a binary observed-label problem, so these classifiers provide probabilities that can rank pages for review. Logistic Regression gives a simple directional reference; the tree gives a readable nonlinear check; and the forest tests whether limited interactions add useful signal. I will choose by held-out **Precision@50**, not by complexity or training performance alone.

The label is `is_declining_label = (trend_direction == "down")`. Because `trend_direction` and `trend_pct` create that label, neither is allowed into the features.

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42

# Find the repository root both locally and when the notebook is opened from its folder.
start = Path.cwd().resolve()
root = next((candidate for candidate in [start, *start.parents] if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists()), None)
if root is None:
    raise FileNotFoundError("Could not find the repository root containing data/raw/content_refresh_anonymized.csv")
raw_path = root / "data" / "raw" / "content_refresh_anonymized.csv"
baseline_path = root / "work" / "notebooks" / "work" / "outputs" / "baseline_action_score.csv"
if not baseline_path.exists():
    alternate = root / "work" / "outputs" / "baseline_action_score.csv"
    baseline_path = alternate if alternate.exists() else baseline_path

df = pd.read_csv(raw_path)
if df.empty:
    raise ValueError("Starter data is empty")
required = {"content_id", "client_id", "trend_direction", "is_declining_label"}
# The raw starter CSV does not always include the prep-added label, so derive it once.
if "is_declining_label" not in df.columns:
    df["is_declining_label"] = df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

print(f"Rows: {len(df):,}; clients: {df['client_id'].nunique():,}; declining rate: {df['is_declining_label'].mean():.3f}")
display(df[["content_id", "client_id", "trend_direction", "is_declining_label"]].head(3))

Rows: 30,000; clients: 32; declining rate: 0.542


,content_id,client_id,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,down,1
1,content_a1fb4e703a9e,client_4e07408562,down,1
2,content_9aa793d4d895,client_7f2253d7e2,down,1


## 2. Split design

The data contains multiple pages per pseudonymous client, so a random row split could let client-specific patterns appear in both train and test. I will hold out 20% of clients using a fixed seed, then verify that both partitions contain both label classes. This estimates performance on unseen clients and keeps the model and the Week 4 baseline on exactly the same test rows. If a client split cannot support both classes, the code uses a stratified row split and reports that fallback explicitly.

In [2]:
def make_client_aware_split(frame, target):
    all_indices = np.arange(len(frame))
    clients = frame["client_id"].fillna("unknown").astype(str)
    unique_clients = clients.drop_duplicates().to_numpy()
    if len(unique_clients) >= 5:
        rng = np.random.default_rng(RANDOM_STATE)
        shuffled = rng.permutation(unique_clients)
        test_client_count = max(1, int(round(len(shuffled) * 0.20)))
        test_clients = set(shuffled[:test_client_count])
        test_mask = clients.isin(test_clients).to_numpy()
        train_idx = all_indices[~test_mask]
        test_idx = all_indices[test_mask]
        if (len(train_idx) and len(test_idx) and target.iloc[train_idx].nunique() == 2 and target.iloc[test_idx].nunique() == 2):
            return train_idx, test_idx, "client_holdout"
    train_idx, test_idx = train_test_split(all_indices, test_size=0.20, random_state=RANDOM_STATE, stratify=target)
    return np.asarray(train_idx), np.asarray(test_idx), "stratified_row_holdout"

target = df["is_declining_label"].astype(int)
train_idx, test_idx, split_strategy = make_client_aware_split(df, target)
train_clients = set(df.iloc[train_idx]["client_id"].astype(str))
test_clients = set(df.iloc[test_idx]["client_id"].astype(str))
print(f"Split: {split_strategy}; train rows: {len(train_idx):,}; test rows: {len(test_idx):,}")
print(f"Train label rate: {target.iloc[train_idx].mean():.3f}; test label rate: {target.iloc[test_idx].mean():.3f}")
if split_strategy == "client_holdout":
    assert train_clients.isdisjoint(test_clients)
assert target.iloc[train_idx].nunique() == 2 and target.iloc[test_idx].nunique() == 2

Split: client_holdout; train rows: 27,675; test rows: 2,325
Train label rate: 0.555; test label rate: 0.391


## 3. Train + compare vs my baseline

The feature matrix uses traffic, content, position, age, freshness, and categorical context available before the label. Missing numeric values are imputed with the training-safe constant 0 and categorical missing values become `unknown`; missingness is also inspected below because it follows content type in this dataset.

The baseline is the Week 4 rule: score `impressions_90d` only when `days_since_last_update >= 180` and `impressions_90d >= 500`, otherwise score 0. Both baseline and models are evaluated only on this notebook's held-out test rows.

In [3]:
numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]
categorical_features = [
    "competition_level", "content_type", "main_intent", "age_tier", "freshness_tier",
    "word_count_tier", "impression_tier", "position_tier",
]
forbidden = {"content_id", "client_id", "trend_direction", "trend_pct", "is_declining_label", "provider_used", "model_used"}
assert forbidden.isdisjoint(set(numeric_features + categorical_features))

numeric = df[[c for c in numeric_features if c in df.columns]].apply(pd.to_numeric, errors="coerce")
numeric = numeric.replace([np.inf, -np.inf], np.nan).fillna(0)
categorical = df[[c for c in categorical_features if c in df.columns]].fillna("unknown").astype(str)
encoded = pd.get_dummies(categorical, prefix=categorical.columns, dtype=float)
X = pd.concat([numeric.reset_index(drop=True), encoded.reset_index(drop=True)], axis=1)
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]
print(f"Feature columns: {X.shape[1]}; missing numeric cells after imputation: {int(X.isna().sum().sum())}")
print("Largest raw missingness rates:")
display(df[[c for c in ["search_volume", "word_count", "avg_position", "main_intent"] if c in df]].isna().mean().sort_values(ascending=False).to_frame("missing_rate"))

models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "decision_tree": DecisionTreeClassifier(class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),
}

def precision_at_k(y_true, scores, k):
    ranked = pd.DataFrame({"y": np.asarray(y_true), "score": np.asarray(scores)}).sort_values("score", ascending=False).head(min(k, len(y_true)))
    return float(ranked["y"].mean()) if len(ranked) else 0.0

def metrics(y_true, scores):
    predictions = (scores >= 0.5).astype(int)
    return {
        "precision_at_20": precision_at_k(y_true, scores, 20),
        "precision_at_50": precision_at_k(y_true, scores, 50),
        "precision_at_100": precision_at_k(y_true, scores, 100),
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "average_precision": float(average_precision_score(y_true, scores)),
        "threshold_precision": float(precision_score(y_true, predictions, zero_division=0)),
        "threshold_recall": float(recall_score(y_true, predictions, zero_division=0)),
    }

baseline_rule_scores = np.where(
    (df.iloc[test_idx]["days_since_last_update"].to_numpy() >= 180)
    & (df.iloc[test_idx]["impressions_90d"].to_numpy() >= 500),
    df.iloc[test_idx]["impressions_90d"].to_numpy(dtype=float),
    0.0,
)
if not baseline_path.exists():
    raise FileNotFoundError(f"Week 4 baseline output not found: {baseline_path}")
baseline_file = pd.read_csv(baseline_path, usecols=["content_id", "score"])
baseline_lookup = baseline_file.set_index("content_id")["score"]
baseline_scores = df.iloc[test_idx]["content_id"].map(baseline_lookup).fillna(0).to_numpy(dtype=float)
assert np.array_equal(baseline_scores, baseline_rule_scores), "Committed Week 4 baseline does not match its documented rule"
comparison = {"baseline_rule": metrics(y_test, baseline_scores)}
trained = {}
probabilities = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    trained[name] = model
    probabilities[name] = model.predict_proba(X_test)[:, 1]
    comparison[name] = metrics(y_test, probabilities[name])

comparison_table = pd.DataFrame(comparison).T[["precision_at_20", "precision_at_50", "precision_at_100", "roc_auc", "average_precision", "threshold_precision", "threshold_recall"]]
best_name = max(models, key=lambda name: (comparison[name]["precision_at_50"], comparison[name]["average_precision"], comparison[name]["roc_auc"]))
print(f"Selected model by held-out Precision@50: {best_name}")
display(comparison_table.round(3))
assert comparison_table.index[0] == "baseline_rule"
assert set(comparison_table.columns) >= {"precision_at_50", "roc_auc"}

Feature columns: 52; missing numeric cells after imputation: 0
Largest raw missingness rates:


,missing_rate
word_count,0.256633
search_volume,0.082267
main_intent,0.079133
avg_position,0.000000


Selected model by held-out Precision@50: random_forest


,precision_at_20,precision_at_50,precision_at_100,roc_auc,average_precision,threshold_precision,threshold_recall
baseline_rule,0.30,0.40,0.37,0.500,0.391,0.000,0.000
logistic_regression,0.45,0.58,0.64,0.725,0.596,0.632,0.571
decision_tree,0.55,0.62,0.60,0.742,0.575,0.569,0.716
random_forest,0.70,0.68,0.70,0.747,0.610,0.560,0.741


### Comparison reading

The table is the decision point: the baseline and learned models use the same held-out rows, the same target, and the same ranking metrics. Precision@50 answers the operational question, ?Among the first 50 pages we would review, how many are observed declines?? A model that only improves a secondary metric or adds complexity without improving this queue metric would not earn adoption.

## 4. Errors and interpretation

I inspect both feature reliance and concrete false positives/false negatives. Permutation importance is measured on the held-out set, so the ranking reflects performance loss when a feature is shuffled rather than only a model-specific coefficient or split count. These are associations in a snapshot; they are not causal explanations.

In [4]:
best_model = trained[best_name]
best_scores = probabilities[best_name]
perm = permutation_importance(best_model, X_test, y_test, n_repeats=5, random_state=RANDOM_STATE, scoring="average_precision", n_jobs=-1)
importance = pd.DataFrame({"feature": X.columns, "importance_mean": perm.importances_mean, "importance_std": perm.importances_std}).sort_values("importance_mean", ascending=False)
print("Top permutation-importance features:")
display(importance.head(10).round(4))

error_frame = df.iloc[test_idx][["content_type", "impressions_90d", "days_since_last_update", "avg_position", "ctr"]].copy().reset_index(drop=True)
error_frame["actual_decline"] = y_test.reset_index(drop=True)
error_frame["predicted_decline"] = (best_scores >= 0.5).astype(int)
error_frame["probability"] = best_scores
error_frame["error_type"] = np.select(
    [
        (error_frame["predicted_decline"] == 1) & (error_frame["actual_decline"] == 0),
        (error_frame["predicted_decline"] == 0) & (error_frame["actual_decline"] == 1),
    ],
    ["false_positive", "false_negative"],
    default="correct",
)
errors = error_frame[error_frame["error_type"] != "correct"].sort_values("probability", ascending=False)
print(f"Held-out errors at probability threshold 0.5: {len(errors):,} of {len(error_frame):,}")
display(errors.head(3).round(3))
print("These examples are hard because the same observed traffic, age, position, and content-type patterns can occur in both declining and non-declining pages; the snapshot does not include every cause of a trend change.")

Top permutation-importance features:


,feature,importance_mean,importance_std
9,days_with_impressions,0.0791,0.0075
5,impressions_90d,0.0287,0.0041
13,ctr,0.0235,0.0069
16,scroll_rate,0.0098,0.0047
0,search_volume,0.0096,0.0014
6,clicks_90d,0.0084,0.0020
14,avg_position,0.0079,0.0054
32,age_tier_365+,0.0034,0.0021
10,days_with_sessions,0.0026,0.0008
15,engagement_rate,0.0025,0.0009


Held-out errors at probability threshold 0.5: 764 of 2,325


,content_type,impressions_90d,days_since_last_update,avg_position,ctr,actual_decline,predicted_decline,probability,error_type
1752,keyword article,5091,20,14.1,0.20,0,1,0.737,false_positive
1778,keyword article,1076,20,25.6,0.09,0,1,0.735,false_positive
1791,keyword article,369,20,21.8,0.00,0,1,0.734,false_positive


These examples are hard because the same observed traffic, age, position, and content-type patterns can occur in both declining and non-declining pages; the snapshot does not include every cause of a trend change.


### Interpretation

The strongest features should be treated as prioritization signals, not levers. Traffic volume and recent activity can plausibly indicate how much evidence exists for a page, while position, CTR, age, and content type describe different review contexts. A false positive is a page that looks like a decline from its available snapshot signals but is labeled otherwise; a false negative is a labeled decline that the model did not rank highly enough. Both error types matter because this queue is decision support, not an automatic refresh command.

## 5. Self-check

- [x] Method choice is tied to the observed binary ranking question.
- [x] The split is client-held out when possible and uses a fixed seed.
- [x] Baseline and models are evaluated on the same rows and metrics.
- [x] Label-derived fields and pseudonymous IDs are excluded from features.
- [x] The comparison table, top features, and concrete errors are shown.
- [x] Claims are observational and decision-support oriented; no causal uplift is claimed.